# Session 3 · Part 2 — Spatial coherence

**Independent checkpoint:** reload data and predictions, compute Moran's I, and save the comparison table and figure.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import morans_i

dataset = load_tutorial_data(paths.raw_data, allow_demo=False)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
common_proteins = dataset.proteins.columns.intersection(predicted.columns)
if common_spots.empty or common_proteins.empty:
    raise ValueError("Observed and predicted data do not share both spot IDs and protein names.")

spots = dataset.spots.loc[common_spots]
observed = dataset.proteins.loc[common_spots, common_proteins]
predicted = predicted.loc[common_spots, common_proteins]


In [ ]:
rows = []
for protein in common_proteins:
    rows.append({
        "protein": protein,
        "observed_morans_i": morans_i(observed[protein], spots),
        "predicted_morans_i": morans_i(predicted[protein], spots),
    })
moran_table = pd.DataFrame(rows)
moran_table


In [ ]:
table_path = paths.results / "session03_morans_i.csv"
figure_path = paths.figures / "session03_morans_i.png"
moran_table.to_csv(table_path, index=False)
sns.scatterplot(data=moran_table, x="observed_morans_i", y="predicted_morans_i", hue="protein", s=80)
plt.axline((0, 0), slope=1, color="black", linewidth=0.8)
plt.title("Observed vs predicted spatial coherence")
plt.tight_layout()
plt.savefig(figure_path, dpi=160)
plt.show()

manifest = write_checkpoint(
    "3.2", [table_path, figure_path],
    summary={"proteins_evaluated": len(moran_table)}, start=paths.root
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

Compare Moran's I with correlation results; the two metrics answer different questions.